# Stage 2 Evaluation - Test Set Metrics

This notebook evaluates the trained Stage 2 models on the test set and computes comprehensive metrics.

**Requirements:**
- Google Colab A100 GPU
- Google Drive with:
  - `/content/drive/MyDrive/LRdataset/test_balanced_npz.zip`
  - `/content/drive/MyDrive/LRdataset/data_split (1).json`
  - `/content/drive/MyDrive/liveness_checkpoints/stage2/` (saved model weights)

**Evaluation Steps:**
1. Load trained Stage 2 model
2. Run inference on test set
3. Compute metrics: Accuracy, Precision, Recall, F1, Specificity, AUC
4. Visualize results: Confusion Matrix, ROC Curve, Score Distributions

## 1. Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q numpy scipy
!pip install -q matplotlib seaborn
!pip install -q scikit-learn
!pip install -q tqdm

In [ ]:
import os
import json
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import signal
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, roc_auc_score
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 2. Extract Test Data

In [ ]:
# Extract test data
zip_path = '/content/drive/MyDrive/LRdataset/test_balanced_npz.zip'
extract_path = '/content/test_balanced_npz/'

if not os.path.exists(extract_path):
    print("Extracting test data...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print(f"Extracted to {extract_path}")
else:
    print(f"Test data already extracted at {extract_path}")

## 3. Model Architecture Definitions

In [ ]:
# Butterworth Filter Bank
class ButterworthFilterBank:
    def __init__(self, fps=30, fc_low=2.0, fc_high=8.0, order=4):
        self.fps = fps
        nyquist = fps / 2.0
        wn_low = fc_low / nyquist
        wn_high = fc_high / nyquist
        self.sos_lf = signal.butter(order, wn_low, btype='lowpass', output='sos')
        self.sos_bp = signal.butter(order, [wn_low, wn_high], btype='bandpass', output='sos')
        self.sos_hf = signal.butter(order, wn_high, btype='highpass', output='sos')

    def apply(self, x):
        T, K, D = x.shape
        x_flat = x.transpose(1, 2, 0).reshape(K * D, T)
        x_lf_flat = signal.sosfiltfilt(self.sos_lf, x_flat, axis=1)
        x_bp_flat = signal.sosfiltfilt(self.sos_bp, x_flat, axis=1)
        x_hf_flat = signal.sosfiltfilt(self.sos_hf, x_flat, axis=1)
        x_lf = x_lf_flat.reshape(K, D, T).transpose(2, 0, 1)
        x_bp = x_bp_flat.reshape(K, D, T).transpose(2, 0, 1)
        x_hf = x_hf_flat.reshape(K, D, T).transpose(2, 0, 1)
        return x_lf, x_bp, x_hf


# TCN Block
class TCNBlock(nn.Module):
    def __init__(self, C_in, C_out, dilation=1, kernel_size=3):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.conv = nn.Conv1d(C_in, C_out, kernel_size=kernel_size, padding=padding, dilation=dilation)
        self.gn = nn.GroupNorm(1, C_out)
        self.act = nn.SiLU()
        self.res = nn.Conv1d(C_in, C_out, 1) if C_in != C_out else nn.Identity()

    def forward(self, x):
        y = self.act(self.gn(self.conv(x)))
        return y + self.res(x)


# Single-Band VAE
class SingleBandVAE(nn.Module):
    def __init__(self, C_in, C_h=48, C_z=12, dilations=[1, 2, 4]):
        super().__init__()
        self.C_in = C_in
        self.C_h = C_h
        self.C_z = C_z

        self.enc_inp = nn.Conv1d(C_in, C_h, 1)
        enc_blocks = [TCNBlock(C_h, C_h, dilation=d) for d in dilations]
        self.encoder = nn.Sequential(*enc_blocks)
        self.enc_out = nn.Conv1d(C_h, C_z * 2, 1)

        self.dec_inp = nn.Conv1d(C_z, C_h, 1)
        dec_blocks = [TCNBlock(C_h, C_h, dilation=d) for d in reversed(dilations)]
        self.decoder = nn.Sequential(*dec_blocks)
        self.dec_out = nn.Conv1d(C_h, C_in, 1)

    def encode(self, x):
        h = self.encoder(self.enc_inp(x))
        mu, logvar = torch.chunk(self.enc_out(h), 2, dim=1)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder(self.dec_inp(z))
        return self.dec_out(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


# Band-Split VAE
class BandSplitVAE(nn.Module):
    def __init__(self, C_in_per_band, C_h=48, C_z=12, dilations=[1, 2, 4]):
        super().__init__()
        self.C_in_per_band = C_in_per_band
        self.vae_lf = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.vae_bp = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.vae_hf = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.register_parameter('fusion_weights', nn.Parameter(torch.ones(3) / 3.0))

    def forward(self, x_lf, x_bp, x_hf):
        x_hat_lf, mu_lf, logvar_lf = self.vae_lf(x_lf)
        x_hat_bp, mu_bp, logvar_bp = self.vae_bp(x_bp)
        x_hat_hf, mu_hf, logvar_hf = self.vae_hf(x_hf)

        weights = F.softmax(self.fusion_weights, dim=0)
        x_hat_fused = weights[0] * x_hat_lf + weights[1] * x_hat_bp + weights[2] * x_hat_hf

        recons = {'lf': x_hat_lf, 'bp': x_hat_bp, 'hf': x_hat_hf}
        mus = {'lf': mu_lf, 'bp': mu_bp, 'hf': mu_hf}
        logvars = {'lf': logvar_lf, 'bp': logvar_bp, 'hf': logvar_hf}

        return recons, mus, logvars, x_hat_fused


# Latent Discriminator (for Method 2)
class LatentDiscriminator(nn.Module):
    def __init__(self, C_z=12, C_h=32):
        super().__init__()
        self.fc1 = nn.Linear(C_z * 3, C_h)
        self.fc2 = nn.Linear(C_h, C_h)
        self.fc3 = nn.Linear(C_h, 1)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, z_lf, z_bp, z_hf):
        z_cat = torch.cat([z_lf, z_bp, z_hf], dim=1)
        z_pooled = z_cat.mean(dim=2)
        h = self.act(self.fc1(z_pooled))
        h = self.act(self.fc2(h))
        logits = self.fc3(h)
        return logits.squeeze(-1)


print("✓ Model architectures defined")

## 4. Feature Engineering Functions

In [ ]:
def compute_velocity(x):
    """Compute velocity using central differences"""
    v = np.zeros_like(x)
    v[1:-1] = (x[2:] - x[:-2]) / 2.0
    v[0] = x[1] - x[0]
    v[-1] = x[-1] - x[-2]
    return v


def compute_acceleration(v):
    """Compute acceleration from velocity"""
    a = np.zeros_like(v)
    a[1:-1] = (v[2:] - v[:-2]) / 2.0
    a[0] = v[1] - v[0]
    a[-1] = v[-1] - v[-2]
    return a


def compute_angle(x):
    """Compute angle from (x, y) coordinates"""
    theta = np.arctan2(x[:, :, 1], x[:, :, 0])
    return theta[..., np.newaxis]


def compute_angle_rate(theta):
    """Compute angular velocity"""
    dtheta = np.zeros_like(theta)
    dtheta[1:-1] = (theta[2:] - theta[:-2]) / 2.0
    dtheta[0] = theta[1] - theta[0]
    dtheta[-1] = theta[-1] - theta[-2]
    return dtheta


def extract_features_simple(x):
    """Extract simple features: position + velocity"""
    v = compute_velocity(x)
    features = np.concatenate([x, v], axis=-1)
    T, K, F = features.shape
    return features.transpose(1, 2, 0).reshape(K * F, T)


def extract_features_full(x):
    """Extract full features: position + velocity + acceleration + angle + angle_rate"""
    v = compute_velocity(x)
    a = compute_acceleration(v)
    theta = compute_angle(x)
    dtheta = compute_angle_rate(theta)
    features = np.concatenate([x, v, a, theta, dtheta], axis=-1)
    T, K, F = features.shape
    return features.transpose(1, 2, 0).reshape(K * F, T)


print("✓ Feature engineering functions defined")

## 5. Test Dataset

In [ ]:
class TestDataset(Dataset):
    def __init__(self, split_json, data_root, feature_mode='simple', T_fixed=150):
        self.data_root = data_root
        self.feature_mode = feature_mode
        self.T_fixed = T_fixed
        self.filter_bank = ButterworthFilterBank(fps=30, fc_low=2.0, fc_high=8.0, order=4)

        # Build file index (filename -> full_path)
        print("Building file index...")
        self.file_index = {}
        for root, dirs, files in os.walk(data_root):
            for file in files:
                if file.endswith('.npz'):
                    full_path = os.path.join(root, file)
                    self.file_index[file] = full_path
        print(f"Found {len(self.file_index)} files in directory")

        # Load split
        with open(split_json, 'r') as f:
            split = json.load(f)['test']

        # Match files using filename only
        self.samples = []
        missing_files = []

        for file_path in split['real']:
            filename = os.path.basename(file_path)
            if filename in self.file_index:
                self.samples.append((self.file_index[filename], 0))
            else:
                missing_files.append(filename)

        for file_path in split['fake']:
            filename = os.path.basename(file_path)
            if filename in self.file_index:
                self.samples.append((self.file_index[filename], 1))
            else:
                missing_files.append(filename)

        if missing_files:
            print(f"⚠ Warning: {len(missing_files)} files not found in directory")
            print(f"  First few missing: {missing_files[:5]}")

        print(f"Loaded {len(self.samples)} test samples")
        n_real = sum(1 for _, label in self.samples if label == 0)
        n_fake = sum(1 for _, label in self.samples if label == 1)
        print(f"  Real: {n_real}, Fake: {n_fake}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        data = np.load(file_path)
        x = data['lips']

        # Pad or crop to T_fixed
        T = x.shape[0]
        if T < self.T_fixed:
            pad = self.T_fixed - T
            x = np.pad(x, ((0, pad), (0, 0), (0, 0)), mode='edge')
        else:
            x = x[:self.T_fixed]

        # Apply frequency decomposition
        x_lf, x_bp, x_hf = self.filter_bank.apply(x)

        # Extract features
        if self.feature_mode == 'simple':
            f_lf = extract_features_simple(x_lf)
            f_bp = extract_features_simple(x_bp)
            f_hf = extract_features_simple(x_hf)
        else:
            f_lf = extract_features_full(x_lf)
            f_bp = extract_features_full(x_bp)
            f_hf = extract_features_full(x_hf)

        return (
            torch.from_numpy(f_lf).float(),
            torch.from_numpy(f_bp).float(),
            torch.from_numpy(f_hf).float(),
            torch.tensor(label, dtype=torch.long)
        )


print("✓ Test dataset class defined")

## 6. Configuration

In [ ]:
# Configuration
FEATURE_MODE = 'simple'  # 'simple' or 'full'
METHOD = 'margin'  # 'margin' or 'discriminator'

# Paths
SPLIT_JSON = '/content/drive/MyDrive/LRdataset/data_split (1).json'
DATA_ROOT = '/content/test_balanced_npz/'
CHECKPOINT_DIR = '/content/drive/MyDrive/liveness_checkpoints/stage2/'

# Model config
K = 40
if FEATURE_MODE == 'simple':
    F_dim = 4
    C_in_per_band = K * F_dim  # 160
else:
    F_dim = 8
    C_in_per_band = K * F_dim  # 320

C_h = 48
C_z = 12
T_fixed = 150

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Feature mode: {FEATURE_MODE}")
print(f"Method: {METHOD}")
print(f"C_in_per_band: {C_in_per_band}")

## 7. Load Test Data

In [ ]:
# Create test dataset
test_dataset = TestDataset(
    split_json=SPLIT_JSON,
    data_root=DATA_ROOT,
    feature_mode=FEATURE_MODE,
    T_fixed=T_fixed
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\nTest set: {len(test_dataset)} samples")
print(f"Number of batches: {len(test_loader)}")

## 8. Load Trained Model

In [ ]:
# Initialize model
model = BandSplitVAE(
    C_in_per_band=C_in_per_band,
    C_h=C_h,
    C_z=C_z,
    dilations=[1, 2, 4]
).to(device)

# Initialize discriminator if using method 2
if METHOD == 'discriminator':
    discriminator = LatentDiscriminator(C_z=C_z, C_h=32).to(device)

# Load checkpoint
if METHOD == 'margin':
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f'method1_margin_{FEATURE_MODE}_best.pt')
else:
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f'method2_discriminator_{FEATURE_MODE}_best.pt')

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if METHOD == 'discriminator' and 'discriminator_state_dict' in checkpoint:
        discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
    print(f"✓ Loaded checkpoint from epoch {checkpoint.get('epoch', 'N/A')}")
    print(f"  Validation accuracy: {checkpoint.get('val_accuracy', 'N/A'):.4f}")
else:
    print(f"⚠ Checkpoint not found: {checkpoint_path}")
    print("Available checkpoints:")
    if os.path.exists(CHECKPOINT_DIR):
        for f in os.listdir(CHECKPOINT_DIR):
            if f.endswith('.pt'):
                print(f"  - {f}")
    else:
        print(f"  Checkpoint directory not found: {CHECKPOINT_DIR}")

model.eval()
if METHOD == 'discriminator':
    discriminator.eval()

print("\n✓ Model loaded and set to evaluation mode")

## 9. Run Inference and Collect Predictions

In [ ]:
def compute_reconstruction_loss(model, x_lf, x_bp, x_hf):
    """Compute reconstruction loss per sample"""
    recons, mus, logvars, _ = model(x_lf, x_bp, x_hf)
    
    # Compute per-band reconstruction loss
    loss_lf = F.l1_loss(recons['lf'], x_lf, reduction='none').mean(dim=[1, 2])
    loss_bp = F.l1_loss(recons['bp'], x_bp, reduction='none').mean(dim=[1, 2])
    loss_hf = F.l1_loss(recons['hf'], x_hf, reduction='none').mean(dim=[1, 2])
    
    # KL divergence
    kl_lf = (-0.5 * (1 + logvars['lf'] - mus['lf'].pow(2) - logvars['lf'].exp())).mean(dim=[1, 2])
    kl_bp = (-0.5 * (1 + logvars['bp'] - mus['bp'].pow(2) - logvars['bp'].exp())).mean(dim=[1, 2])
    kl_hf = (-0.5 * (1 + logvars['hf'] - mus['hf'].pow(2) - logvars['hf'].exp())).mean(dim=[1, 2])
    
    # Total reconstruction loss
    total_loss = loss_lf + loss_bp + loss_hf + kl_lf + kl_bp + kl_hf
    
    return total_loss, mus


# Run inference
print("Running inference on test set...")

all_labels = []
all_scores = []

with torch.no_grad():
    for x_lf, x_bp, x_hf, labels in tqdm(test_loader, desc="Evaluating"):
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        
        if METHOD == 'margin':
            # Method 1: Use reconstruction loss as score
            scores, _ = compute_reconstruction_loss(model, x_lf, x_bp, x_hf)
        else:
            # Method 2: Use discriminator output as score
            recons, mus, logvars, _ = model(x_lf, x_bp, x_hf)
            logits = discriminator(mus['lf'], mus['bp'], mus['hf'])
            scores = torch.sigmoid(logits)
        
        all_labels.extend(labels.cpu().numpy())
        all_scores.extend(scores.cpu().numpy())

all_labels = np.array(all_labels)
all_scores = np.array(all_scores)

print(f"\n✓ Inference complete")
print(f"  Total samples: {len(all_labels)}")
print(f"  Real samples: {(all_labels == 0).sum()}")
print(f"  Fake samples: {(all_labels == 1).sum()}")
print(f"  Score range: [{all_scores.min():.4f}, {all_scores.max():.4f}]")

## 10. Determine Optimal Threshold

In [ ]:
# For margin method, lower score = fake (smoother motion)
# For discriminator method, higher score = fake

if METHOD == 'margin':
    # Invert scores for margin method (so higher = more likely fake)
    scores_for_roc = -all_scores
else:
    scores_for_roc = all_scores

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(all_labels, scores_for_roc)
roc_auc = auc(fpr, tpr)

# Find optimal threshold (Youden's J statistic)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold_roc = thresholds[optimal_idx]

# Convert back to original score space if needed
if METHOD == 'margin':
    optimal_threshold = -optimal_threshold_roc
else:
    optimal_threshold = optimal_threshold_roc

print(f"\n{'='*60}")
print(f"ROC Analysis")
print(f"{'='*60}")
print(f"AUC: {roc_auc:.4f}")
print(f"Optimal threshold: {optimal_threshold:.4f}")
print(f"  TPR at optimal: {tpr[optimal_idx]:.4f}")
print(f"  FPR at optimal: {fpr[optimal_idx]:.4f}")
print(f"{'='*60}")

## 11. Compute Metrics

In [ ]:
# Make predictions using optimal threshold
if METHOD == 'margin':
    predictions = (all_scores < optimal_threshold).astype(int)
else:
    predictions = (all_scores > optimal_threshold).astype(int)

# Compute metrics
accuracy = accuracy_score(all_labels, predictions)
precision = precision_score(all_labels, predictions, zero_division=0)
recall = recall_score(all_labels, predictions, zero_division=0)
f1 = f1_score(all_labels, predictions, zero_division=0)

# Compute confusion matrix
cm = confusion_matrix(all_labels, predictions)
tn, fp, fn, tp = cm.ravel()

# Compute additional metrics
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0
fpr_final = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

# Print results
print(f"\n{'='*60}")
print(f"Test Set Evaluation Results")
print(f"{'='*60}")
print(f"Method: {METHOD.upper()}")
print(f"Feature Mode: {FEATURE_MODE.upper()}")
print(f"Threshold: {optimal_threshold:.4f}")
print(f"")
print(f"Primary Metrics:")
print(f"  Accuracy:    {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision:   {precision:.4f} ({precision*100:.2f}%)")
print(f"  Recall:      {recall:.4f} ({recall*100:.2f}%)")
print(f"  F1-Score:    {f1:.4f}")
print(f"  AUC-ROC:     {roc_auc:.4f}")
print(f"")
print(f"Additional Metrics:")
print(f"  Specificity: {specificity:.4f} ({specificity*100:.2f}%)")
print(f"  NPV:         {npv:.4f} ({npv*100:.2f}%)")
print(f"  FPR:         {fpr_final:.4f} ({fpr_final*100:.2f}%)")
print(f"  FNR:         {fnr:.4f} ({fnr*100:.2f}%)")
print(f"")
print(f"Confusion Matrix:")
print(f"  TN: {tn:4d}  |  FP: {fp:4d}")
print(f"  FN: {fn:4d}  |  TP: {tp:4d}")
print(f"{'='*60}")

## 12. Visualizations

In [ ]:
# Create figure with subplots
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# 1. Confusion Matrix
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
ax1.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax1.set_ylabel('True Label', fontsize=12)
ax1.set_xlabel('Predicted Label', fontsize=12)

# 2. ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax2.scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', s=100, 
           label=f'Optimal (TPR={tpr[optimal_idx]:.3f}, FPR={fpr[optimal_idx]:.3f})', zorder=5)
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.3)

# 3. Score Distribution
ax3 = fig.add_subplot(gs[0, 2])
real_scores = all_scores[all_labels == 0]
fake_scores = all_scores[all_labels == 1]

ax3.hist(real_scores, bins=50, alpha=0.6, label='Real', color='green', density=True)
ax3.hist(fake_scores, bins=50, alpha=0.6, label='Fake', color='red', density=True)
ax3.axvline(optimal_threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold={optimal_threshold:.3f}')
ax3.set_xlabel('Score', fontsize=12)
ax3.set_ylabel('Density', fontsize=12)
ax3.set_title('Score Distribution', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Metrics Bar Chart
ax4 = fig.add_subplot(gs[1, 0])
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'Specificity']
metrics_values = [accuracy, precision, recall, f1, specificity]
colors_bar = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
bars = ax4.bar(metrics_names, metrics_values, color=colors_bar, alpha=0.7)
ax4.set_ylim([0, 1.0])
ax4.set_ylabel('Score', fontsize=12)
ax4.set_title('Performance Metrics', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, metrics_values):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

# 5. Box Plot
ax5 = fig.add_subplot(gs[1, 1])
bp_data = [real_scores, fake_scores]
bp = ax5.boxplot(bp_data, labels=['Real', 'Fake'], patch_artist=True,
                 boxprops=dict(facecolor='lightblue', alpha=0.7),
                 medianprops=dict(color='red', linewidth=2))
ax5.axhline(optimal_threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold={optimal_threshold:.3f}')
ax5.set_ylabel('Score', fontsize=12)
ax5.set_title('Score Distribution by Class', fontsize=14, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3, axis='y')

# 6. Summary Text
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
summary_text = f"""
EVALUATION SUMMARY
{'='*40}

Method: {METHOD.upper()}
Feature Mode: {FEATURE_MODE.upper()}

Dataset:
  Total samples: {len(all_labels)}
  Real samples:  {(all_labels == 0).sum()}
  Fake samples:  {(all_labels == 1).sum()}

Performance:
  Accuracy:      {accuracy:.4f}
  Precision:     {precision:.4f}
  Recall:        {recall:.4f}
  F1-Score:      {f1:.4f}
  AUC-ROC:       {roc_auc:.4f}
  Specificity:   {specificity:.4f}

Threshold: {optimal_threshold:.4f}

Score Statistics:
  Real: {real_scores.mean():.4f} ± {real_scores.std():.4f}
  Fake: {fake_scores.mean():.4f} ± {fake_scores.std():.4f}
"""
ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes,
        fontsize=11, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle(f'Stage 2 Test Set Evaluation - {METHOD.upper()} Method ({FEATURE_MODE.upper()} features)',
            fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

# Save figure
save_path = f'/content/drive/MyDrive/liveness_checkpoints/stage2/evaluation_{METHOD}_{FEATURE_MODE}.png'
fig.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Visualization saved to: {save_path}")

## 13. Per-Class Score Statistics

In [ ]:
print(f"\n{'='*60}")
print(f"Per-Class Score Statistics")
print(f"{'='*60}")
print(f"\nReal Samples (n={len(real_scores)}):")
print(f"  Mean:   {real_scores.mean():.4f}")
print(f"  Std:    {real_scores.std():.4f}")
print(f"  Min:    {real_scores.min():.4f}")
print(f"  Max:    {real_scores.max():.4f}")
print(f"  Median: {np.median(real_scores):.4f}")
print(f"  Q1:     {np.percentile(real_scores, 25):.4f}")
print(f"  Q3:     {np.percentile(real_scores, 75):.4f}")

print(f"\nFake Samples (n={len(fake_scores)}):")
print(f"  Mean:   {fake_scores.mean():.4f}")
print(f"  Std:    {fake_scores.std():.4f}")
print(f"  Min:    {fake_scores.min():.4f}")
print(f"  Max:    {fake_scores.max():.4f}")
print(f"  Median: {np.median(fake_scores):.4f}")
print(f"  Q1:     {np.percentile(fake_scores, 25):.4f}")
print(f"  Q3:     {np.percentile(fake_scores, 75):.4f}")

# Separation measure
from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(real_scores, fake_scores)
print(f"\nSeparation Analysis:")
print(f"  T-statistic: {t_stat:.4f}")
print(f"  P-value:     {p_value:.4e}")
print(f"  Significant: {'Yes' if p_value < 0.05 else 'No'} (α=0.05)")
print(f"{'='*60}")

## 14. Save Results

In [ ]:
# Save detailed results
results = {
    'method': METHOD,
    'feature_mode': FEATURE_MODE,
    'threshold': float(optimal_threshold),
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'auc_roc': float(roc_auc),
        'specificity': float(specificity),
        'npv': float(npv),
        'fpr': float(fpr_final),
        'fnr': float(fnr)
    },
    'confusion_matrix': {
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp)
    },
    'score_statistics': {
        'real': {
            'mean': float(real_scores.mean()),
            'std': float(real_scores.std()),
            'min': float(real_scores.min()),
            'max': float(real_scores.max()),
            'median': float(np.median(real_scores)),
            'q1': float(np.percentile(real_scores, 25)),
            'q3': float(np.percentile(real_scores, 75))
        },
        'fake': {
            'mean': float(fake_scores.mean()),
            'std': float(fake_scores.std()),
            'min': float(fake_scores.min()),
            'max': float(fake_scores.max()),
            'median': float(np.median(fake_scores)),
            'q1': float(np.percentile(fake_scores, 25)),
            'q3': float(np.percentile(fake_scores, 75))
        }
    },
    'dataset': {
        'total_samples': len(all_labels),
        'real_samples': int((all_labels == 0).sum()),
        'fake_samples': int((all_labels == 1).sum())
    }
}

# Save to JSON
results_path = f'/content/drive/MyDrive/liveness_checkpoints/stage2/evaluation_{METHOD}_{FEATURE_MODE}.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to: {results_path}")

# Also save predictions for further analysis
predictions_data = {
    'labels': all_labels.tolist(),
    'scores': all_scores.tolist(),
    'predictions': predictions.tolist()
}

predictions_path = f'/content/drive/MyDrive/liveness_checkpoints/stage2/predictions_{METHOD}_{FEATURE_MODE}.json'
with open(predictions_path, 'w') as f:
    json.dump(predictions_data, f)

print(f"✓ Predictions saved to: {predictions_path}")
print(f"\n{'='*60}")
print(f"EVALUATION COMPLETE")
print(f"{'='*60}")